# 🎬 AutoScene Studio — GPU Renderer (Colab)
Runs your **existing AutoScene web app** on a **Colab GPU** (NVENC hardware
encoding = ~10x faster), with **Edge / Fish / Kokoro** voices, saves finished
videos to **Google Drive**, and exposes the dashboard via a **Cloudflare
tunnel**. No login required.

## How to use
1. Run each cell in order (▶ Run all). A popup may ask to grant **Drive access**
   — click Allow (that's where videos get saved).
2. After the **Cloudflare** cell you'll see a `https://xxx.trycloudflare.com` URL
   — **open it**. That's your AutoScene dashboard.
3. Upload media + script, pick a voice (Edge/Fish/Kokoro), hit render. Finished
   video saves to `MyDrive/AutoScene_Outputs/`.
4. The **last cell plays silent audio** so Colab doesn't disconnect mid-render
   (it keeps the tab "active"). Keep that tab open.


---
## Step 1 — Setup: Drive + repos + engine venv + GPU ffmpeg


In [ ]:

# Mount Google Drive (finished videos are saved here)
from google.colab import drive
drive.mount('/content/drive')
import os
print("Drive mounted:", os.path.exists('/content/drive/MyDrive'))


In [ ]:

import os, subprocess
print("AutoScene GPU notebook — v5")
# --- clone the working engine + web app ---
os.chdir('/content')
def clone(url, dest):
    if not os.path.exists(dest):
        subprocess.run(['git','clone','-q','--depth','1',url,dest], check=True)
        print("cloned", dest)
clone('https://github.com/5656wwed/automan-engine', '/content/automan-engine')
clone('https://github.com/5656wwed/automan-web-colab', '/content/automan-web')
print("repos ready")


In [ ]:

# --- engine deps: install into base Colab Python (a venv is not needed here) ---
import os, subprocess, sys
os.chdir('/content')
PYV = sys.executable                       # Colab's base python (has torch)
open('/content/PYV.txt', 'w').write(PYV)   # used by the launch cell later
deps = ['edge-tts', 'openai', 'aiohttp', 'pydub', 'Pillow',
        'python-dotenv', 'typer', 'rich', 'pydantic', 'requests',
        'fastapi', 'uvicorn']
r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q'] + deps,
                   capture_output=True, text=True)
if r.returncode != 0:
    print('⚠ some deps failed to install (see below) — continuing anyway')
    print(r.stderr[-500:])
print('interpreter:', PYV)
print('deps installed')


In [ ]:

# --- ffmpeg: use the SYSTEM (apt) build, like the dubbing — it loads libcuda
#     for NVENC, whereas the static BtbN build cannot (Cannot load libcuda.so.1).
#     GPU is auto-detected: if a GPU is present, use h264_nvenc; else libx264. ---
import os, shutil, subprocess
os.chdir('/content')
try:
    has_gpu = subprocess.run(['nvidia-smi'], capture_output=True).returncode == 0
except FileNotFoundError:
    has_gpu = False  # CPU-only runtime: nvidia-smi not installed
print('GPU (nvidia-smi):', has_gpu)
ff = shutil.which('ffmpeg') or '/usr/bin/ffmpeg'
fp = shutil.which('ffprobe') or '/usr/bin/ffprobe'
if not os.path.exists(ff):
    subprocess.run('apt-get install -y -qq ffmpeg', shell=True, check=False)
    ff = shutil.which('ffmpeg') or '/usr/bin/ffmpeg'
print('ffmpeg:', subprocess.run([ff, '-version'], capture_output=True, text=True).stdout.splitlines()[0])
enc = subprocess.run([ff, '-hide_banner', '-encoders'], capture_output=True, text=True).stdout
nvenc = enc.count('nvenc')
# NVENC self-test (only meaningful if a GPU is present)
_t = subprocess.run([ff, '-v', 'error', '-f', 'lavfi', '-i',
                     'testsrc=size=320x180:rate=15', '-frames:v', '1',
                     '-c:v', 'h264_nvenc', '-y', '/tmp/_nv_test.mp4'],
                    capture_output=True, text=True)
works = has_gpu and os.path.exists('/tmp/_nv_test.mp4')
open('/content/nvenc_ok', 'w').write('1' if (works and nvenc) else '0')
open('/content/FFMPEG_PATH', 'w').write(ff)
open('/content/FFPROBE_PATH', 'w').write(fp)
print('nvenc encoders:', nvenc, '| NVENC self-test:', 'WORKS ✅' if works else 'FAILS ❌')
if has_gpu and not works:
    print((_t.stderr or '').strip()[-400:])
print('encoder:', 'h264_nvenc (GPU)' if works and nvenc else 'libx264 (CPU)')
print('using:', ff)


In [ ]:
# --- patch the engine: NVENC encode + Kokoro provider + audio controls ---
# patch_engine.py is taken from the freshly-cloned repo (kept in sync, no stale copy).
import subprocess, os, shutil
os.chdir('/content')
PATCH_SRC = '/content/automan-web/patch_engine.py'
if not os.path.exists(PATCH_SRC):
    raise FileNotFoundError(f'{PATCH_SRC} missing - did the clone step run?')
shutil.copy(PATCH_SRC, '/content/patch_engine.py')
r = subprocess.run(['python3','/content/patch_engine.py','/content/automan-engine'], capture_output=True, text=True)
print(r.stdout); print(r.stderr[-400:] if r.returncode else "OK")


---
## Step 2 — Start Kokoro TTS (GPU) + the web app + Cloudflare tunnel


In [ ]:
# --- Kokoro TTS server (needs Python <3.13; Colab base is 3.13, so use a 3.11 venv via uv) ---
# kokoro_server.py is taken from the freshly-cloned repo (kept in sync).
import os, subprocess, sys, time, requests, shutil
os.chdir('/content')
KOKORO_SRC_PATH = '/content/automan-web/kokoro_server.py'
if not os.path.exists(KOKORO_SRC_PATH):
    raise FileNotFoundError(f'{KOKORO_SRC_PATH} missing - did the clone step run?')
shutil.copy(KOKORO_SRC_PATH, '/content/kokoro_server.py')

kokoro_ok = False
try:
    # ensure uv (preinstalled on Colab, but be safe)
    if not shutil.which('uv'):
        subprocess.run([sys.executable,'-m','pip','install','-q','uv'], check=True)
    # Kokoro's misaki dep requires Python <3.13 -> dedicated 3.11 venv
    if not os.path.exists('/content/kokoro-venv'):
        subprocess.run(['uv','venv','--python','3.11','/content/kokoro-venv'], check=True,
                       capture_output=True)
    kok_py = '/content/kokoro-venv/bin/python'
    r = subprocess.run([kok_py,'-m','pip','install','-q','kokoro>=0.9.4','soundfile',
                        'fastapi','uvicorn'], capture_output=True, text=True)
    if r.returncode == 0:
        subprocess.run('apt-get -qq -y install espeak-ng > /dev/null 2>&1', shell=True)
        kokoro_ok = True
        print('kokoro installed in 3.11 venv')
    else:
        print('kokoro pip failed:', (r.stderr or '')[-800:])
except Exception as e:
    print('kokoro setup error:', e)

if kokoro_ok:
    ffmpeg_for_kokoro = open('/content/FFMPEG_PATH').read().strip()
    env = dict(os.environ); env['KOKORO_FFMPEG']=ffmpeg_for_kokoro; env['KOKORO_PORT']='8002'
    open('/content/kokoro.log','w').close()
    p = subprocess.Popen([kok_py,'/content/kokoro_server.py'], env=env,
                         stdout=open('/content/kokoro.log','a'), stderr=subprocess.STDOUT, cwd='/content')
    for _ in range(120):
        try:
            r = requests.get('http://localhost:8002/v1/models', timeout=2)
            if r.status_code == 200: break
        except Exception: pass
        time.sleep(3)
    print("Kokoro server up:", os.path.exists('/content/kokoro.log'))
    print(open('/content/kokoro.log').read()[-300:])
else:
    print('WARNING: Kokoro TTS skipped (Python 3.13 incompatibility) - Edge/Fish voices still work.')


In [ ]:

# --- launch the AutoScene web app on a local port ---
import os, subprocess, time, requests
WEB = '/content/automan-web'
PYV = open('/content/PYV.txt').read().strip()  # engine interpreter (venv or base)
os.makedirs('/content/drive/MyDrive/AutoScene_Outputs', exist_ok=True)
ffmpeg_bin = open('/content/FFMPEG_PATH').read().strip()
ffprobe_bin = open('/content/FFPROBE_PATH').read().strip()
# codec=auto -> engine probes real 1-frame encodes and picks the best
# working hardware encoder (h264_nvenc > h264_qsv > h264_amf > libx264).
# NVENC self-test result is printed for info below.
env = dict(os.environ)
env.update({
  'THEAUTOMAN_DIR': '/content/automan-engine',
  'THEAUTOMAN_PYTHON': PYV,
'AUTOMAN_CODEC': 'auto',
  'FFMPEG_PATH': ffmpeg_bin,
  'FFPROBE_PATH': ffprobe_bin,
  'DRIVE_OUT_DIR': '/content/drive/MyDrive/AutoScene_Outputs',
  'KOKORO_URL': 'http://localhost:8002/v1',
  'POCKET_VOICES_DIR': '/content/pocket_voices',
  'LOG_PATH': '/content/webapp.log',
})
print('ffmpeg:', ffmpeg_bin)
print('encoder: auto-detect (engine picks best working; NVENC self-test was:',
      'WORKS' if open('/content/nvenc_ok').read().strip() == '1' else 'FAILS/CPU')
open('/content/webapp.log','w').close()
p = subprocess.Popen([PYV,'-m','uvicorn','app.main:app','--host','0.0.0.0','--port','7860'],
                     cwd=WEB, env=env, stdout=open('/content/webapp.log','a'), stderr=subprocess.STDOUT)
for _ in range(60):
    try:
        r = requests.get('http://localhost:7860/health', timeout=2)
        if r.status_code == 200: break
    except Exception: pass
    time.sleep(2)
print("web app health:", requests.get('http://localhost:7860/health', timeout=5).status_code)
print(open('/content/webapp.log').read()[-300:])


In [ ]:

# --- Cloudflare tunnel -> public dashboard URL ---
import subprocess, time, re
# install cloudflared if missing
subprocess.run(['which','cloudflared'], capture_output=True)
import shutil
if not shutil.which('cloudflared'):
    subprocess.run(['wget','-q','https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64','-O','/usr/local/bin/cloudflared'], check=True)
    subprocess.run(['chmod','+x','/usr/local/bin/cloudflared'])
subprocess.run(['pkill','-f','cloudflared'], capture_output=True)
time.sleep(1)
open('/content/tunnel.log','w').close()
p = subprocess.Popen(['cloudflared','tunnel','--url','http://localhost:7860','--no-autoupdate'],
                     stdout=open('/content/tunnel.log','a'), stderr=subprocess.STDOUT)
url = None
for _ in range(40):
    txt = open('/content/tunnel.log', errors='ignore').read()
    m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', txt)
    if m: url = m.group(0); break
    time.sleep(3)
print("\n\n🖥  DASHBOARD URL:", url or "(still connecting... check /content/tunnel.log)")
print("Open it in your browser — no login needed.")


---
## Step 3 — Keep the session alive (silence audio so Colab doesn't disconnect)


In [ ]:
%%javascript
// Play silent audio on a loop so Colab keeps the tab 'active' and
// doesn't disconnect the session during long renders.
(function(){
  var b64 = '...';
  var a = new Audio('data:audio/wav;base64,'+b64);
  a.loop = true; a.volume = 0.01;
  a.play();
  setInterval(function(){ if(a.paused){ a.play(); } }, 30000);
  console.log('Keep-alive silence audio playing');
})();
